# Phase 4 — Baseline 1: Rule-Based Routing

Yen's K-shortest-paths algorithm over the maritime graph. Accepts a `weight_key` parameter so the same function can rank routes using either:
- `base_weight` (pure distance, ignores disruption)
- `disrupted_weight` (distance + disruption penalty from Phase 3)

This lets us show disrupted vs non-disrupted routing side by side for the demo.

In [97]:
import pickle
import networkx as nx
import pandas as pd
from itertools import islice

PROCESSED_DIR = "../data/processed"

# Load the disruption-tagged graph (has both base_weight and disrupted_weight on every edge)
with open(f"{PROCESSED_DIR}/maritime_graph_disrupted.gpickle", "rb") as f:
    G = pickle.load(f)

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Quick sanity check: confirm both weight attributes exist on edges
sample_edge = list(G.edges(data=True))[0]
print("Sample edge attrs:", sample_edge[2])

Graph loaded: 1543 nodes, 9462 edges
Sample edge attrs: {'distance_nm': 175.2965283650975, 'base_weight': 175.2965283650975, 'disrupted_weight': 175.2965283650975, 'disruption_severity_edge': 0.0}


In [98]:
def port_lookup_by_name(G, name_substring):
    """Helper: find port_id(s) by partial name match, for picking origin/destination during testing."""
    matches = [
        (node_id, attrs['port_name'], attrs['country'])
        for node_id, attrs in G.nodes(data=True)
        if name_substring.lower() in attrs['port_name'].lower()
    ]
    return matches

# Example usage
port_lookup_by_name(G, "singapore")

[(50000, 'Keppel - (East Singapore)', 'Singapore')]

In [99]:
def k_shortest_paths(G, origin_id, dest_id, k=5, weight_key='disrupted_weight'):
    """
    Returns the K shortest paths between origin_id and dest_id using the specified
    edge weight attribute ('base_weight' or 'disrupted_weight').

    Uses NetworkX's shortest_simple_paths generator (Yen's algorithm equivalent),
    which yields paths in increasing order of weight.
    """
    if origin_id not in G:
        raise ValueError(f"Origin port_id {origin_id} not found in graph")
    if dest_id not in G:
        raise ValueError(f"Destination port_id {dest_id} not found in graph")
    if not nx.has_path(G, origin_id, dest_id):
        raise ValueError(f"No path exists between {origin_id} and {dest_id}")

    paths_generator = nx.shortest_simple_paths(G, origin_id, dest_id, weight=weight_key)

    results = []
    for path in islice(paths_generator, k):
        total_weight = sum(
            G[path[i]][path[i+1]][weight_key] for i in range(len(path) - 1)
        )
        total_distance_nm = sum(
            G[path[i]][path[i+1]]['distance_nm'] for i in range(len(path) - 1)
        )
        max_edge_disruption = max(
            (G[path[i]][path[i+1]].get('disruption_severity_edge', 0.0) for i in range(len(path) - 1)),
            default=0.0
        )
        results.append({
            'path': path,
            'path_port_names': [G.nodes[p]['port_name'] for p in path],
            'num_hops': len(path) - 1,
            'total_weight': round(total_weight, 2),
            'total_distance_nm': round(total_distance_nm, 2),
            'max_disruption_exposure': round(max_edge_disruption, 3),
            'weight_key_used': weight_key
        })

    return results

In [100]:
def print_routes(routes, title="Routes"):
    print(f"\n=== {title} ===")
    for i, r in enumerate(routes, 1):
        route_str = " -> ".join(r['path_port_names'])
        print(f"{i}. [{r['num_hops']} hops, {r['total_distance_nm']} nm, "
              f"weight={r['total_weight']}, disruption_exposure={r['max_disruption_exposure']}]")
        print(f"   {route_str}")

## Test: pick an origin-destination pair that passes near a disrupted region
Using the Phase 3 fallback scenario (Red Sea / Suez), a good test pair is a port on the Europe side vs a port on the Asia side — a real voyage would normally transit the Red Sea/Suez corridor.

In [101]:
# Find candidate origin/destination near Europe and Asia
print("Europe candidates:", port_lookup_by_name(G, "rotterdam"))
print("Asia candidates:", port_lookup_by_name(G, "singapore"))

Europe candidates: [(31140, 'Rotterdam', 'Netherlands')]
Asia candidates: [(50000, 'Keppel - (East Singapore)', 'Singapore')]


In [102]:
# --- Set these based on the lookup results above ---
ORIGIN_ID = 31140   # <-- fill in from port_lookup_by_name output
DEST_ID = 50000     # <-- fill in from port_lookup_by_name output

assert ORIGIN_ID is not None and DEST_ID is not None, "Set ORIGIN_ID and DEST_ID from the lookup above before running"

routes_disrupted = k_shortest_paths(G, ORIGIN_ID, DEST_ID, k=5, weight_key='disrupted_weight')
print_routes(routes_disrupted, title="Disrupted routing (avoids penalized edges)")

routes_baseline = k_shortest_paths(G, ORIGIN_ID, DEST_ID, k=5, weight_key='base_weight')
print_routes(routes_baseline, title="Non-disrupted routing (pure shortest distance)")


=== Disrupted routing (avoids penalized edges) ===
1. [46 hops, 8875.64 nm, weight=13852.73, disruption_exposure=0.755]
   Rotterdam -> Zeebrugge -> Oostende -> Boulogne-Sur-Mer -> Fecamp -> Port De Caen -> Saint-Malo -> Nantes -> Le Verdon -> Bayonne -> Tarragona -> Palamos -> Toulon -> Imperia -> Portoferraio -> Civitavecchia -> Formia -> Torre Annunziata -> Trani -> Brindisi -> Kerkira -> Volos -> Ormos Aliveriou -> Neon Karlovas -> Antalya -> Larnaca -> Bayrut -> Ashdod -> Elat -> Duba -> Jiddah -> Mitsiwa Harbor -> Ras Isa Marine Terminal -> Al Mukalla -> Mina Raysut -> Mina Al Fahl -> Gwadar -> Mundra -> Marmagao -> Kattupalli Port -> Gopalpur -> Bassein -> Mergui -> Phuket -> Pulau Pinang -> Melaka -> Keppel - (East Singapore)
2. [47 hops, 8875.64 nm, weight=13852.73, disruption_exposure=0.755]
   Rotterdam -> Zeebrugge -> Oostende -> Boulogne-Sur-Mer -> Fecamp -> Port De Caen -> Saint-Malo -> Nantes -> Le Verdon -> Bayonne -> Tarragona -> Palamos -> Toulon -> Imperia -> Portof

## Side-by-side comparison
If disruption is working correctly, the top disrupted-route recommendation should differ from the top non-disrupted route (or have higher distance but lower disruption exposure), for pairs whose shortest path passes near a disrupted zone.

In [103]:
comparison = pd.DataFrame([
    {
        'rank': i+1,
        'disrupted_route': " -> ".join(routes_disrupted[i]['path_port_names']) if i < len(routes_disrupted) else None,
        'disrupted_distance_nm': routes_disrupted[i]['total_distance_nm'] if i < len(routes_disrupted) else None,
        'disrupted_exposure': routes_disrupted[i]['max_disruption_exposure'] if i < len(routes_disrupted) else None,
        'baseline_route': " -> ".join(routes_baseline[i]['path_port_names']) if i < len(routes_baseline) else None,
        'baseline_distance_nm': routes_baseline[i]['total_distance_nm'] if i < len(routes_baseline) else None,
    }
    for i in range(max(len(routes_disrupted), len(routes_baseline)))
])
comparison

,rank,disrupted_route,disrupted_distance_nm,disrupted_exposure,baseline_route,baseline_distance_nm
0,1,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8875.64,0.755,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.80
1,2,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8875.64,0.755,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.81
2,3,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8875.65,0.755,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.81
3,4,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8875.65,0.755,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.82
4,5,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8875.65,0.755,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.83


In [104]:
# --- Save Baseline 1 as a reusable module for the API (Phase 6) ---
# This cell content will be converted into src/ranking/baseline.py

print("Baseline 1 functions ready: k_shortest_paths(), port_lookup_by_name()")
print("These will be moved into src/ranking/baseline.py for reuse in the API.")

Baseline 1 functions ready: k_shortest_paths(), port_lookup_by_name()
These will be moved into src/ranking/baseline.py for reuse in the API.


diagnosis

In [105]:
for r in routes_disrupted:
    print(r['path'])
    print(r['path_port_names'])
    print("---")

[31140, 31280, 31310, 35760, 35830, 35880, 36130, 36900, 37120, 37230, 38540, 38580, 38870, 39370, 39720, 39810, 39865, 39990, 40570, 40500, 41720, 42470, 42340, 42840, 44820, 44960, 45030, 45100, 48076, 48106, 48140, 47900, 48155, 48210, 48230, 48255, 48590, 48617, 48970, 49454, 49500, 49640, 49690, 49770, 49850, 49970, 50000]
['Rotterdam', 'Zeebrugge', 'Oostende', 'Boulogne-Sur-Mer', 'Fecamp', 'Port De Caen', 'Saint-Malo', 'Nantes', 'Le Verdon', 'Bayonne', 'Tarragona', 'Palamos', 'Toulon', 'Imperia', 'Portoferraio', 'Civitavecchia', 'Formia', 'Torre Annunziata', 'Trani', 'Brindisi', 'Kerkira', 'Volos', 'Ormos Aliveriou', 'Neon Karlovas', 'Antalya', 'Larnaca', 'Bayrut', 'Ashdod', 'Elat', 'Duba', 'Jiddah', 'Mitsiwa Harbor', 'Ras Isa Marine Terminal', 'Al Mukalla', 'Mina Raysut', 'Mina Al Fahl', 'Gwadar', 'Mundra', 'Marmagao', 'Kattupalli Port', 'Gopalpur', 'Bassein', 'Mergui', 'Phuket', 'Pulau Pinang', 'Melaka', 'Keppel - (East Singapore)']
---
[31140, 31280, 31310, 35760, 35830, 35880

In [106]:
top_route = routes_disrupted[0]['path']
for port_id in top_route:
    sev = G.nodes[port_id].get('disruption_severity', 0.0)
    if sev > 0:
        print(f"DISRUPTED: {G.nodes[port_id]['port_name']} — severity {sev}")
print("Full route:", [G.nodes[p]['port_name'] for p in top_route])

DISRUPTED: Ashdod — severity 0.705
DISRUPTED: Elat — severity 0.705
DISRUPTED: Duba — severity 0.705
DISRUPTED: Ras Isa Marine Terminal — severity 0.755
Full route: ['Rotterdam', 'Zeebrugge', 'Oostende', 'Boulogne-Sur-Mer', 'Fecamp', 'Port De Caen', 'Saint-Malo', 'Nantes', 'Le Verdon', 'Bayonne', 'Tarragona', 'Palamos', 'Toulon', 'Imperia', 'Portoferraio', 'Civitavecchia', 'Formia', 'Torre Annunziata', 'Trani', 'Brindisi', 'Kerkira', 'Volos', 'Ormos Aliveriou', 'Neon Karlovas', 'Antalya', 'Larnaca', 'Bayrut', 'Ashdod', 'Elat', 'Duba', 'Jiddah', 'Mitsiwa Harbor', 'Ras Isa Marine Terminal', 'Al Mukalla', 'Mina Raysut', 'Mina Al Fahl', 'Gwadar', 'Mundra', 'Marmagao', 'Kattupalli Port', 'Gopalpur', 'Bassein', 'Mergui', 'Phuket', 'Pulau Pinang', 'Melaka', 'Keppel - (East Singapore)']


In [107]:
routes_disrupted_extended = k_shortest_paths(G, ORIGIN_ID, DEST_ID, k=15, weight_key='disrupted_weight')
for r in routes_disrupted_extended:
    print(r['num_hops'], r['total_distance_nm'], r['max_disruption_exposure'])

46 8875.64 0.755
47 8875.64 0.755
47 8875.65 0.755
46 8875.65 0.755
47 8875.65 0.755
47 8875.66 0.755
48 8875.66 0.755
48 8875.66 0.755
48 8875.66 0.755
46 8875.67 0.755
47 8875.67 0.755
47 8875.67 0.755
47 8875.67 0.755
48 8875.67 0.755
47 8875.67 0.755


In [108]:
for r in routes_disrupted_extended:
    print(r['path'])
    print(r['path_port_names'])
    print("---")

[31140, 31280, 31310, 35760, 35830, 35880, 36130, 36900, 37120, 37230, 38540, 38580, 38870, 39370, 39720, 39810, 39865, 39990, 40570, 40500, 41720, 42470, 42340, 42840, 44820, 44960, 45030, 45100, 48076, 48106, 48140, 47900, 48155, 48210, 48230, 48255, 48590, 48617, 48970, 49454, 49500, 49640, 49690, 49770, 49850, 49970, 50000]
['Rotterdam', 'Zeebrugge', 'Oostende', 'Boulogne-Sur-Mer', 'Fecamp', 'Port De Caen', 'Saint-Malo', 'Nantes', 'Le Verdon', 'Bayonne', 'Tarragona', 'Palamos', 'Toulon', 'Imperia', 'Portoferraio', 'Civitavecchia', 'Formia', 'Torre Annunziata', 'Trani', 'Brindisi', 'Kerkira', 'Volos', 'Ormos Aliveriou', 'Neon Karlovas', 'Antalya', 'Larnaca', 'Bayrut', 'Ashdod', 'Elat', 'Duba', 'Jiddah', 'Mitsiwa Harbor', 'Ras Isa Marine Terminal', 'Al Mukalla', 'Mina Raysut', 'Mina Al Fahl', 'Gwadar', 'Mundra', 'Marmagao', 'Kattupalli Port', 'Gopalpur', 'Bassein', 'Mergui', 'Phuket', 'Pulau Pinang', 'Melaka', 'Keppel - (East Singapore)']
---
[31140, 31280, 31310, 35760, 35830, 35880

In [109]:
# Temporarily strip out all edges touching any disrupted port, then check if Rotterdam-Singapore is still reachable at all
G_test = G.copy()
disrupted_edges = [(u, v) for u, v, d in G_test.edges(data=True) if d.get('disruption_severity_edge', 0) > 0]
G_test.remove_edges_from(disrupted_edges)

print(f"Removed {len(disrupted_edges)} disrupted edges")
print("Path still exists avoiding disruption entirely?", nx.has_path(G_test, ORIGIN_ID, DEST_ID))

if nx.has_path(G_test, ORIGIN_ID, DEST_ID):
    bypass_path = nx.shortest_path(G_test, ORIGIN_ID, DEST_ID, weight='base_weight')
    bypass_dist = nx.shortest_path_length(G_test, ORIGIN_ID, DEST_ID, weight='base_weight')
    print(f"Bypass distance: {bypass_dist} nm")
    print([G.nodes[p]['port_name'] for p in bypass_path])
else:
    print("NO bypass exists in this graph at all — confirms the Cape route is missing from the graph, not just unranked.")

Removed 292 disrupted edges
Path still exists avoiding disruption entirely? True
Bypass distance: 14322.789810576158 nm
['Rotterdam', 'Zeebrugge', 'Oostende', 'Boulogne-Sur-Mer', 'Fecamp', 'Port De Caen', 'Saint-Malo', 'Nantes', 'Le Verdon', 'Santander', 'Aviles', 'Aveiro', 'Lagos', 'Safi', 'Agadir', 'Nouakchott', 'Conakry', 'Tema', 'Pennington Oil Terminal', 'Port Owendo', 'Takula Terminal', 'Palanca Terminal', 'Walvis Bay', 'Luderitz Bay', 'Richards Bay', 'Beira', 'Dar Es Salaam', 'Muqdisho', 'Boosaaso', 'Mina Raysut', 'Mina Al Fahl', 'Gwadar', 'Mundra', 'Marmagao', 'Kattupalli Port', 'Gopalpur', 'Bassein', 'Mergui', 'Phuket', 'Pulau Pinang', 'Melaka', 'Keppel - (East Singapore)']


In [110]:
# Check how many ports exist along the West African coast in our filtered graph
west_africa_ports = [
    (node_id, attrs['port_name'], attrs['country'], attrs['latitude'], attrs['longitude'])
    for node_id, attrs in G.nodes(data=True)
    if -20 <= attrs['latitude'] <= 35 and -20 <= attrs['longitude'] <= 15
]
print(f"West African region ports in graph: {len(west_africa_ports)}")
for p in sorted(west_africa_ports, key=lambda x: -x[3]):
    print(p)

West African region ports in graph: 50
(45390, 'Mersa Sfax', 'Tunisia', 34.733333, 10.766667)
(45785, 'Kenitra', 'Morocco', 34.3, -6.6)
(45377, 'Ashtart Oil Terminal', 'Tunisia', 34.283333, 11.383333)
(45375, 'Gabes', 'Tunisia', 33.9, 10.116667)
(45368, 'Didon Terminal', 'Tunisia', 33.783333, 11.9)
(45793, 'Casablanca', 'Morocco', 33.6, -7.616667)
(45796, 'El Jorf Lasfar', 'Morocco', 33.116667, -8.616667)
(45330, 'Mina Tarabulus (Tripoli)', 'Libya', 32.9, 13.183333)
(45335, 'Az Zawiya', 'Libya', 32.8, 12.716667)
(38130, 'Funchal', 'Portugal', 32.633333, -16.916667)
(45797, 'Safi', 'Morocco', 32.3, -9.25)
(45802, 'Agadir', 'Morocco', 30.433333, -9.633333)
(38160, 'Santa Cruz De Tenerife', 'Spain', 28.466667, -16.233333)
(38170, 'Las Palmas', 'Spain', 28.15, -15.416667)
(45812, 'Nouadhibou', 'Mauritania', 20.916667, -17.05)
(45814, 'Nouakchott', 'Mauritania', 18.033333, -16.033333)
(45818, 'St Louis', 'Senegal', 16.016667, -16.516667)
(45820, 'Dakar', 'Senegal', 14.683333, -17.433333)
(4

In [111]:
# Check if Cape Town / South Africa area is even reachable from West Africa in this graph
capetown_matches = port_lookup_by_name(G, "cape town")
print("Cape Town matches:", capetown_matches)

if capetown_matches:
    ct_id = capetown_matches[0][0]
    dakar_matches = port_lookup_by_name(G, "dakar")
    print("Dakar matches:", dakar_matches)
    if dakar_matches:
        dk_id = dakar_matches[0][0]
        print("Path exists Dakar->Cape Town?", nx.has_path(G, dk_id, ct_id))

Cape Town matches: [(46770, 'Cape Town', 'South Africa')]
Dakar matches: [(45820, 'Dakar', 'Senegal')]
Path exists Dakar->Cape Town? True


In [112]:
# Direct distance check: Namibe -> Cape Town gap
namibe_id = 46610
capetown_id = 46770
print("Direct edge exists Namibe->CapeTown?", G_test.has_edge(namibe_id, capetown_id))
if nx.has_path(G_test, namibe_id, capetown_id):
    print("Namibe->CapeTown distance:", nx.shortest_path_length(G_test, namibe_id, capetown_id, weight='base_weight'))
    print([G.nodes[p]['port_name'] for p in nx.shortest_path(G_test, namibe_id, capetown_id, weight='base_weight')])

# Full route cost via Cape (force it through Cape Town as a waypoint)
dist_rotterdam_to_capetown = nx.shortest_path_length(G_test, ORIGIN_ID, capetown_id, weight='base_weight')
dist_capetown_to_singapore = nx.shortest_path_length(G_test, capetown_id, DEST_ID, weight='base_weight')
print(f"Rotterdam->CapeTown: {dist_rotterdam_to_capetown} nm")
print(f"CapeTown->Singapore: {dist_capetown_to_singapore} nm")
print(f"Total via Cape: {dist_rotterdam_to_capetown + dist_capetown_to_singapore} nm")

Direct edge exists Namibe->CapeTown? False
Namibe->CapeTown distance: 1173.9774014642962
['Namibe', 'Walvis Bay', 'Cape Town']
Rotterdam->CapeTown: 6613.999110433558 nm
CapeTown->Singapore: 7969.65384203248 nm
Total via Cape: 14583.652952466036 nm


In [113]:
# Walk the actual shortest path Cape Town -> Singapore and look for suspiciously long individual hops
path_ct_sg = nx.shortest_path(G_test, capetown_id, DEST_ID, weight='base_weight')
print("Full path:")
for i in range(len(path_ct_sg) - 1):
    u, v = path_ct_sg[i], path_ct_sg[i+1]
    hop_dist = G_test[u][v]['base_weight']
    print(f"{G.nodes[u]['port_name']} -> {G.nodes[v]['port_name']}: {hop_dist:.1f} nm")

Full path:
Cape Town -> Maputo: 875.2 nm
Maputo -> Beira: 389.1 nm
Beira -> Dar Es Salaam: 823.7 nm
Dar Es Salaam -> Muqdisho: 643.3 nm
Muqdisho -> Boosaaso: 600.5 nm
Boosaaso -> Mina Raysut: 442.0 nm
Mina Raysut -> Mina Al Fahl: 474.0 nm
Mina Al Fahl -> Gwadar: 225.6 nm
Gwadar -> Mundra: 431.3 nm
Mundra -> Marmagao: 496.4 nm
Marmagao -> Kattupalli Port: 401.9 nm
Kattupalli Port -> Gopalpur: 447.4 nm
Gopalpur -> Bassein: 577.6 nm
Bassein -> Mergui: 344.5 nm
Mergui -> Phuket: 276.4 nm
Phuket -> Pulau Pinang: 185.9 nm
Pulau Pinang -> Melaka: 224.2 nm
Melaka -> Keppel - (East Singapore): 110.7 nm


In [114]:
# Sanity-check the actual penalty math on the Ras Isa hop vs a Cape route hop
for u, v, d in G.edges(data=True):
    if G.nodes[u]['port_name'] in ['Ras Isa Marine Terminal', 'Al Mukalla', 'Mitsiwa Harbor'] or \
       G.nodes[v]['port_name'] in ['Ras Isa Marine Terminal', 'Al Mukalla', 'Mitsiwa Harbor']:
        print(f"{G.nodes[u]['port_name']} -> {G.nodes[v]['port_name']}: base={d['base_weight']:.1f}, disrupted={d['disrupted_weight']:.1f}, severity={d.get('disruption_severity_edge', 0)}")

Al Khair Oil Terminal -> Mitsiwa Harbor: base=269.8, disrupted=269.8, severity=0.0
Al Khair Oil Terminal -> Ras Isa Marine Terminal: base=406.6, disrupted=1941.5, severity=0.755
Berbera -> Ras Isa Marine Terminal: base=314.8, disrupted=1503.0, severity=0.755
Berbera -> Al Mukalla: base=342.6, disrupted=1635.8, severity=0.755
Beshayer Oil Terminal -> Mitsiwa Harbor: base=258.4, disrupted=258.4, severity=0.0
Beshayer Oil Terminal -> Ras Isa Marine Terminal: base=396.6, disrupted=1894.0, severity=0.755
Boosaaso -> Al Mukalla: base=194.2, disrupted=194.2, severity=0.0
Boosaaso -> Ras Isa Marine Terminal: base=448.8, disrupted=2143.1, severity=0.755
Al Mukha -> Ras Isa Marine Terminal: base=115.4, disrupted=551.3, severity=0.755
Al Mukha -> Mitsiwa Harbor: base=259.7, disrupted=1240.0, severity=0.755
Al Mukha -> Al Mukalla: base=349.4, disrupted=1668.3, severity=0.755
Ras Isa Marine Terminal -> Al Ahmadi: base=27.9, disrupted=133.1, severity=0.755
Ras Isa Marine Terminal -> Jizan: base=106.